# The AI Maturity Ladder: Five Form Factors Of AI Applications

Climb from a plain LLM powered chatbot to an autonomous agent that writes and runs its own code — one rung at a time.

## What You Will Build

A single running example — a support assistant for a fictional product called **Acme Cloud** — implemented five different ways, each one more capable than the last:

1. **The Chatbot** — a stateless LLM that answers from its training knowledge alone.
2. **RAG** — the same LLM, now grounded in Acme Cloud's actual documentation (stored in Oracle AI Database).
3. **The Workflow** — several LLM calls stitched together by *your code* into a reliable pipeline.
4. **The Agent** — an LLM that decides *for itself* which tools to call and when, looping until the job is done.
5. **The Autonomous Agent** — an agent that doesn't just answer, but **writes and runs code** to build durable automation.

Each rung adds exactly one new capability. By the end you'll have a clear mental model of *which* form factor to reach for, and *why*.

## What You Will Learn

- Call Claude for a single-shot response and a multi-turn conversation with the `anthropic` SDK
- Ground a model in your own data with a Retrieval-Augmented Generation (RAG) pipeline on **Oracle AI Database**
- Work through Oracle's **retrieval techniques** — keyword, vector, attribute filtering (pre/in/post), hybrid (RRF), and graph
- Orchestrate multiple Claude calls into a deterministic, code-controlled **workflow** (classify → retrieve → draft → review)
- Build a true **agent** with the `claude-agent-sdk` that chooses its own tools
- Let an agent **build automation** — write a script to disk, run it, and fix it autonomously
- Decide which form factor fits a given problem

## The Ladder at a Glance

| # | Form Factor | Who controls the flow? | Data access | Can act on the world? |
|---|-------------|------------------------|-------------|------------------------|
| 1 | **Chatbot** | You (one call) | Training data only | No |
| 2 | **RAG** | You (retrieve → generate) | + Your documents (Oracle) | No |
| 3 | **Workflow** | **Your code** (fixed steps) | + Your documents | Within your pipeline |
| 4 | **Agent** | **The model** (dynamic) | + Tools you provide | Via tools |
| 5 | **Autonomous Agent** | **The model** (dynamic) | + Files & shell | Yes — writes & runs code |


![The five form factors](../images/form_factors.png)

![One use case, five ways](../images/form_factors_use_case.png)


> 💡 **Key Insight — Add one capability at a time:** Each rung of this ladder differs from the one below it by a *single* new power. Naming that difference is the whole point — it tells you the simplest form factor that can solve your problem. Reaching for an autonomous agent when RAG would do is the most common (and expensive) mistake in applied AI.

This taxonomy mirrors Anthropic's guidance in [*Building Effective Agents*](https://www.anthropic.com/research/building-effective-agents): start simple, and add autonomy only when the task genuinely needs it.

## Setup

> **Note:** The cell below installs everything this notebook needs. If your environment already has these packages (for example, a prepared workshop Codespace), you can skip it.
>
> We use **`fastembed`** for embeddings — it runs the model through ONNX (`onnxruntime`) and needs **no PyTorch**, which keeps the notebook portable across platforms and Python versions where a torch wheel may not exist.

In [1]:
%pip install -Uq anthropic oracledb fastembed pandas numpy claude-agent-sdk nest_asyncio

Note: you may need to restart the kernel to use updated packages.


## Configure API Access

Every form factor authenticates with a single environment variable, `ANTHROPIC_API_KEY`. Both the `anthropic` SDK and the `claude-agent-sdk` read it automatically. Set it in your shell (or a `.env` you load) before running — or just run the next cell, which securely prompts you for the key with `getpass` if it isn't already set.

In [ ]:
import os
import getpass
import anthropic

# Both SDKs read ANTHROPIC_API_KEY from the environment automatically.
# If it isn't already set (via your shell or a .env), prompt for it here — getpass
# hides the input so the key never shows up in the notebook output.
if not os.environ.get("ANTHROPIC_API_KEY"):
    os.environ["ANTHROPIC_API_KEY"] = getpass.getpass("Enter your ANTHROPIC_API_KEY: ")

# One model string, used everywhere. claude-opus-4-8 is the latest, most capable Claude.
MODEL = "claude-opus-4-8"
MAX_TOKENS = 1024  # these are short, demo-sized answers; raise for longer outputs

client = anthropic.Anthropic()
print("Anthropic client ready ✓")

Anthropic client ready ✓


### Smoke-Test the Client

A single round-trip confirms the key works and introduces the shape of every Claude call: `model`, `max_tokens`, an optional `system` prompt, and a list of `messages`.

In [3]:
response = client.messages.create(
    model=MODEL,
    max_tokens=MAX_TOKENS,
    system="You are a concise, friendly assistant.",
    messages=[{"role": "user", "content": "In one sentence, what is Claude?"}],
)

# response.content is a list of typed blocks; for a plain answer we read the first text block.
print(response.content[0].text)

Claude is an AI assistant developed by Anthropic, designed to be helpful, harmless, and honest in conversations and tasks.


One tiny helper we'll reuse everywhere: it pulls the plain text out of a response, which can contain several block types (text, thinking, tool-use, …).

In [4]:
def text_of(response) -> str:
    """Concatenate the text blocks of a Claude response into a single string."""
    return "".join(block.text for block in response.content if block.type == "text")


print(text_of(response))

Claude is an AI assistant developed by Anthropic, designed to be helpful, harmless, and honest in conversations and tasks.


# Form Factor 1 — The Chatbot

> 💡 **Key Term — LLM (Large Language Model):** A model trained to predict text. On its own it has no memory between calls, no access to your data, and no ability to take actions. It maps a prompt to a response — nothing more, nothing less.

The simplest useful thing you can build with Claude: send a question, get an answer. No database, no tools, no retrieval. This is where almost every AI project starts.

![Anatomy of a chatbot](../images/ff1_anatomy.png)

![LLMs are stateless](../images/ff1_stateless.png)

![Memory = re-sending the whole conversation](../images/ff1_resending_whole_conversation.png)


### A single prompt in, a single response out

`ask_claude` wraps one `messages.create` call. That's the entire chatbot.

In [5]:
def ask_claude(question: str, system: str = "You are a helpful assistant.") -> str:
    """Form Factor 1: a single prompt in, a single response out. Nothing else."""
    response = client.messages.create(
        model=MODEL,
        max_tokens=MAX_TOKENS,
        system=system,
        messages=[{"role": "user", "content": question}],
    )
    return text_of(response)


print(ask_claude("Explain what an API rate limit is, in two sentences."))

An API rate limit is a restriction set by an API provider that caps the number of requests a client can make within a specific time period (such as 100 requests per minute). It helps prevent server overload, ensures fair usage among users, and protects against abuse like denial-of-service attacks.


## 1.1 Adding Memory — by Resending the Conversation

The API is **stateless**: Claude remembers nothing between calls. A "chatbot" feels like it remembers only because *we* resend the full conversation history on every turn.

> 💡 **Key Term — Statelessness:** Each request is independent. To hold a conversation, you accumulate the `messages` list yourself and send all of it every time. The model's "memory" is whatever you put back in front of it.

The `Chatbot` class below turns the single-shot call into a **multi-turn conversation**. Because the API is stateless, the class keeps its own `history` list and re-sends it on every turn.

- **`send(user_message)`** is the heart of it: it appends the user's message to `history`, calls `client.messages.create(...)` with the *entire* history, extracts the reply, and appends that reply back to `history` before returning it.
- **`self.history.append({...})`** is how "memory" is built. Each turn adds two entries — one `{"role": "user", ...}` and one `{"role": "assistant", ...}` — so the next call sees the whole conversation so far.
- **`system`** is the *system prompt*: a standing instruction that sets the assistant's role and behaviour. It is sent separately from the conversation `messages` and is not part of the back-and-forth turns.
- **`max_tokens`** caps how many tokens Claude may **generate** in its reply — it limits the *output* length, not the input. We set it once as `MAX_TOKENS` and reuse it; raise it when you need longer answers.

In [6]:
class Chatbot:
    """A multi-turn chatbot. The 'memory' is just a growing list of messages we resend."""

    def __init__(self, system: str = "You are a helpful assistant."):
        self.system = system
        self.history: list[dict] = []

    def send(self, user_message: str) -> str:
        self.history.append({"role": "user", "content": user_message})
        response = client.messages.create(
            model=MODEL,
            max_tokens=MAX_TOKENS,
            system=self.system,
            messages=self.history,  # the entire conversation, every time
        )

        reply = text_of(response)

        # Here is where the history of the conversation is maintained via appending the response this turn
        self.history.append({"role": "assistant", "content": reply})

        return reply

Below we start a conversation: we instantiate a `Chatbot` and send it our first message. This is **Turn 1** — the bot stores both our message and its reply in `history`, so the next turn has this context to draw on.

In [7]:
bot = Chatbot()
print("Turn 1:", bot.send("Hi! My name is Priya and I run data infrastructure."))


Turn 1: Hi Priya! Nice to meet you. Data infrastructure is a fascinating (and demanding) area to work in.

What can I help you with today? Whether it's something like:

- **Architecture decisions** (batch vs. streaming, data lake vs. warehouse, etc.)
- **Tooling** (Airflow, dbt, Kafka, Spark, Snowflake, etc.)
- **Scaling/performance** challenges
- **Data quality, governance, or observability**
- **Cost optimization**
- **Team/process questions**
- Or just thinking through a problem out loud

...I'm happy to dig in. What's on your plate right now?


In [8]:
print("\nTurn 2:", bot.send("What's my name and what do I do?"))


Turn 2: Your name is Priya, and you run data infrastructure. 😊

Anything you'd like to dive into?


> 💡 **Teachable moment:** Turn 2 only works because Turn 1 is still in `bot.history`. Every turn re-sends the whole conversation, so cost and latency grow with the conversation length — which is exactly why later form factors add retrieval and context management instead of just stuffing everything into the history forever.

To chat interactively yourself, define and call the loop below.

In [9]:
def chat_session():
    """Interactive REPL-style chat. Type 'exit' to stop."""
    session = Chatbot()
    print("Chatting with Claude — type 'exit' to quit.\n")
    while True:
        user = input("You: ")
        if user.strip().lower() in {"exit", "quit"}:
            break
        print("Claude:", session.send(user), "\n")


# Uncomment to try it (uses input()):
# chat_session()

## 1.2 The Ceiling of a Chatbot

A chatbot can only draw on what it learned during training. Ask it something specific to *your* world — internal docs, private data, a product it has never seen — and it simply can't know.

In [10]:
print(ask_claude(
    "What is the exact API rate limit, in requests per minute, "
    "on Acme Cloud's Pro plan?"
))

I don't have specific, reliable information about Acme Cloud's Pro plan API rate limits—and I'm not certain "Acme Cloud" refers to a real, specific product (it's sometimes used as a generic placeholder name).

To get the exact number, I'd recommend:

1. **Checking their official API documentation** — rate limits are almost always documented there, often under "Rate Limits," "Quotas," or "Usage Limits."
2. **Logging into your account dashboard** — limits are sometimes shown per-plan there.
3. **Contacting their support or sales team** — especially if limits aren't published.
4. **Inspecting response headers** — many APIs return headers like `X-RateLimit-Limit` and `X-RateLimit-Remaining` that tell you your current limits.

If you can point me to the specific company/product you mean (or share their docs), I'm happy to help you interpret the rate limit details. I just don't want to make up a number that could be wrong.


> 💡 **Key Insight — The data gap:** Claude has never seen Acme Cloud's documentation, so it can't answer with specifics — at best it explains the gap, at worst it guesses. The fix isn't a bigger model; it's *giving the model the relevant data at question time*. That is exactly what the next rung — RAG — does.

> ### 📌 Key Takeaways — Form Factor 1
> - A chatbot is **one stateless `messages.create` call**: prompt in, text out.
> - "Memory" is an illusion you create by **re-sending the whole conversation** each turn.
> - Its hard ceiling: **no access to your data** and **knowledge frozen at training time** — which motivates every rung above.

# Form Factor 2 — Retrieval-Augmented Generation (RAG)

> 💡 **Key Term — RAG (Retrieval-Augmented Generation):** A pattern where you first *retrieve* the documents relevant to a question, then hand them to the LLM alongside the question. The model answers from real, current data instead of relying on training knowledge alone.

This time we put the knowledge base in a real database — **Oracle AI Database** — store Acme Cloud's documentation as native `VECTOR` embeddings, and work through the retrieval techniques Oracle supports: **keyword**, **vector**, **attribute filtering** (pre / in / post), **hybrid** (keyword + vector), and **graph**. Retrieval quality is the single biggest lever on RAG quality, so it is worth seeing the options side by side.

![Retrieval-Augmented Generation](../images/ff2_rag.png)

![Anatomy of RAG](../images/ff2_anatomy.png)


> 💡 **Key Term — Converged database:** Oracle AI Database stores relational data, full-text indexes, vector embeddings, and graphs together in one engine. That lets us run keyword search, semantic vector search, attribute-filtered and hybrid combinations, and graph traversals — all in SQL, against a single set of tables.

### Set up Oracle locally (Docker)

Form Factor 2 needs a real Oracle AI Database. This repo ships a one-command starter — **`oracle.sh`**, in the *parent* folder (next to `appbook/`) so the notebook **and** the production app share one database — built on the [`gvenzl/oracle-free`](https://hub.docker.com/r/gvenzl/oracle-free) image (Oracle Database 23ai Free).

**Prerequisite:** [Docker](https://docs.docker.com/get-docker/) installed and running.

From a terminal, in the `ai_maturity_form_factors/` folder (one level up from this notebook):

```bash
./oracle.sh start
```

…or run it straight from a notebook cell: `!../oracle.sh start`. The first run pulls the image (~2 GB) and creates the database, so give it a few minutes. It is idempotent — re-run it any time; `status`, `logs`, `sql`, `stop`, and `remove` manage it.

The script provisions **exactly** what the techniques below rely on:

| What it sets up | Used by |
|---|---|
| A `VECTOR` user at `localhost:1521/FREEPDB1` (matches `appbook/.env.example`) | the connection in §2.1 |
| **Oracle Text** (`CTXSYS.CONTEXT` index) | keyword & hybrid search — §2.6.1 / §2.6.4 |
| **AI Vector Search** memory pool | so the HNSW vector index builds — §2.4 / §2.6.3 |
| **`CREATE PROPERTY GRAPH`** privilege | graph retrieval — §2.6.5 |


> **Already have an Oracle AI Database?** Skip the script — just ensure a `VECTOR` user is reachable at `localhost:1521/FREEPDB1`, overriding the `ORACLE_USER` / `ORACLE_PASSWORD` / `ORACLE_DSN` environment variables the next cell reads if yours differ. Prepared workshop environments already provision all of the above.

<details>
<summary><b>Prefer to run Docker by hand, without the script?</b></summary>

```bash
docker run -d --name acme-oracle-free \
  -p 1521:1521 \
  -e ORACLE_PASSWORD=OraclePwd_2025 \
  -e APP_USER=VECTOR \
  -e APP_USER_PASSWORD=VectorPwd_2025 \
  -v acme-oracle-data:/opt/oracle/oradata \
  gvenzl/oracle-free:23
```

Once the container is healthy, grant the extras the notebook needs (vector pool + property graph), then restart so the vector pool activates:

```bash
docker exec -i acme-oracle-free sqlplus -S / as sysdba <<'SQL'
ALTER SYSTEM SET vector_memory_size = 512M SCOPE=SPFILE;
ALTER SESSION SET CONTAINER = FREEPDB1;
GRANT CREATE SESSION, RESOURCE, CREATE VIEW, CREATE PROPERTY GRAPH, CTXAPP TO VECTOR;
ALTER USER VECTOR QUOTA UNLIMITED ON USERS;
SQL
docker restart acme-oracle-free
```

</details>

## 2.1 Connect to Oracle AI Database

A small retry helper, then we keep the connection in `conn` for the rest of the notebook.

In [11]:
import oracledb
import time


def connect_to_oracle(max_retries=3, retry_delay=5):
    """Connect to the local Oracle AI Database, retrying a few times on startup."""
    user = os.environ.get("ORACLE_USER", "VECTOR")
    password = os.environ.get("ORACLE_PASSWORD", "VectorPwd_2025")
    dsn = os.environ.get("ORACLE_DSN", "localhost:1521/FREEPDB1")
    for attempt in range(1, max_retries + 1):
        try:
            print(f"Connection attempt {attempt}/{max_retries}...")
            conn = oracledb.connect(user=user, password=password, dsn=dsn)
            with conn.cursor() as cur:
                cur.execute("SELECT banner FROM v$version WHERE banner LIKE 'Oracle%'")
                print("Connected:", cur.fetchone()[0])
            return conn
        except oracledb.OperationalError as e:
            print(f"Attempt {attempt} failed: {e}")
            if attempt < max_retries:
                time.sleep(retry_delay)
            else:
                raise


conn = connect_to_oracle()

Connection attempt 1/3...
Connected: Oracle AI Database 26ai Free Release 23.26.0.0.0 - Develop, Learn, and Run for Free


## 2.2 The Knowledge Base

Acme Cloud's documentation — a dozen short docs across categories (billing, API, security, data, …). Each becomes one row in Oracle; the `category` field will drive both **attribute filtering** (§2.6.3) and **graph retrieval** (§2.6.5).

**The shape of the data.** `DOCS` is a plain Python list of dictionaries — one dict per document — and every dict carries the same four string fields:

| Field | Role | Example |
|---|---|---|
| `doc_id` | Stable unique key — becomes the table's `PRIMARY KEY`. | `"sso"` |
| `title` | Human-readable name; embedded together with the body for richer retrieval. | `"Single Sign-On (SSO)"` |
| `category` | A value from a small controlled vocabulary (`billing`, `api`, `data`, `security`, `support`, …). Drives metadata filtering and graph edges. | `"security"` |
| `content` | The doc body — one to three sentences of documentation prose. | `"SSO is available on the Enterprise plan…"` |

A fifth field, `embedding` (a 768-dim vector), is **not** written by hand — we compute it from `title + content` in §2.3 and store it as a table column in §2.4. The data is **synthetic**: short, self-consistent docs written for this lesson so the retrieval behaviour is easy to reason about.

In [12]:
DOCS = [
    {"doc_id": "plans", "title": "Plans & Pricing", "category": "billing",
     "content": "Acme Cloud offers three plans. Free includes 1 project and community support. "
                "Pro is $49 per user per month with email support and advanced analytics. "
                "Enterprise has custom pricing, SSO, and a dedicated support engineer."},
    {"doc_id": "rate_limits", "title": "API Rate Limits", "category": "api",
     "content": "Acme Cloud enforces API rate limits per plan. The Free plan allows 60 requests "
                "per minute. The Pro plan allows 1,000 requests per minute. Enterprise rate limits "
                "are negotiated per contract."},
    {"doc_id": "upgrade", "title": "Upgrading Your Plan", "category": "billing",
     "content": "To upgrade, open Settings, then Billing, then Change Plan. Upgrades take effect "
                "immediately and are pro-rated for the current billing cycle."},
    {"doc_id": "regions", "title": "Data Residency & Regions", "category": "data",
     "content": "Acme Cloud stores data in US, EU (Frankfurt), and APAC (Singapore) regions. "
                "The region is chosen at project creation and cannot be changed afterward."},
    {"doc_id": "sla", "title": "Service Level Agreement", "category": "reliability",
     "content": "Acme Cloud guarantees 99.9% uptime on the Pro plan and 99.99% on Enterprise. "
                "SLA credits are issued automatically when monthly uptime falls below target."},
    {"doc_id": "support", "title": "Support Channels & Response Times", "category": "support",
     "content": "Free plan customers use the community forum. Pro email support responds within one "
                "business day. Enterprise includes 24/7 support with a one-hour response target for "
                "critical issues."},
    {"doc_id": "api_keys", "title": "Creating & Rotating API Keys", "category": "api",
     "content": "Create and rotate API keys under Settings, then API Keys. Rotating a key immediately "
                "revokes the previous one, so update your applications before rotating."},
    {"doc_id": "sso", "title": "Single Sign-On (SSO)", "category": "security",
     "content": "SSO is available on the Enterprise plan and supports SAML 2.0 and OIDC. An "
                "administrator configures the identity provider under Settings, then Security, then SSO."},
    {"doc_id": "webhooks", "title": "Webhooks", "category": "integrations",
     "content": "Acme Cloud can send webhooks on project events. Configure endpoint URLs under "
                "Settings, then Webhooks. Failed deliveries are retried with exponential backoff for "
                "up to 24 hours."},
    {"doc_id": "backups", "title": "Backups & Recovery", "category": "data",
     "content": "Acme Cloud takes automated daily backups retained for 30 days on Pro and 90 days on "
                "Enterprise. Point-in-time recovery is available on the Enterprise plan."},
    {"doc_id": "roles", "title": "Team Roles & Permissions", "category": "account",
     "content": "Acme Cloud supports Owner, Admin, Member, and Viewer roles. Only Owners and Admins "
                "can manage billing, invite teammates, or rotate API keys."},
    {"doc_id": "export", "title": "Exporting Your Data", "category": "data",
     "content": "You can export project data as JSON or CSV from Settings, then Export. Large exports "
                "are emailed as a downloadable archive when ready."},
]
print(len(DOCS), "documents in the knowledge base.")

12 documents in the knowledge base.


## 2.3 Embed the Documents

> 💡 **Key Term — Embedding:** A numeric vector that captures the meaning of text. Texts with similar meaning have similar vectors, which lets us search by *meaning* rather than exact keywords.

We use `nomic-embed-text-v1.5` via **fastembed** (ONNX — no PyTorch). It's an *asymmetric* model: documents and queries are embedded with different prefixes, which fastembed applies for us — `embed()` for documents, `query_embed()` for queries.

> **Note:** The first call downloads the model (a few hundred MB) and may take a minute.

![From text to vectors (embeddings)](../images/ff2_text_to_vectors.png)

![Chunking documents](../images/ff2_chunking.png)

![Four ways to chunk](../images/ff2_chunking_strategies.png)


**Why normalize?** The `_unit` helper divides each embedding by its length (its L2 norm) so every vector has magnitude 1. Two things follow:

- **Cosine similarity becomes a plain dot product.** Cosine measures the *angle* between two vectors and ignores their length; once vectors are unit-length, their dot product *is* the cosine. That keeps the similarity math simple and consistent.
- **Every vector sits on the same scale.** Comparisons aren't skewed by one document happening to produce a longer raw vector than another — only *direction* (meaning) matters.

Oracle's `VECTOR_DISTANCE(..., COSINE)` normalizes internally too, so this mainly keeps our own client-side math — the graph similarity edges in §2.6.5 — clean and unambiguous.

In [13]:
from fastembed import TextEmbedding
import numpy as np

embedder = TextEmbedding(model_name="nomic-ai/nomic-embed-text-v1.5")


def _unit(v):
    """Normalize to a unit vector so cosine similarity is a plain dot product."""
    v = np.asarray(v, dtype=np.float32)
    n = np.linalg.norm(v)
    return v / n if n else v


/Users/richmondalake/opt/anaconda3/envs/oracle_demos/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


**How `doc_texts` is built.** For each document we concatenate two fields — the **title** and the **content** — into a single string with `f"{d['title']}. {d['content']}"`, then embed that one string to get one vector per document.

**Why concatenate several fields?** An embedding turns *one* piece of text into *one* vector, so everything we want a document to be findable by has to live in that text:

- The **title** carries dense, high-signal keywords ("API Rate Limits", "Single Sign-On") that pull the vector toward the document's topic.
- The **content** adds the detail and surrounding context that lets semantic, paraphrased queries match.

Joining them with `". "` gives the model a clean, sentence-like input and a richer representation than either field alone — so a query can match on the title's keywords *or* the body's meaning. Fields we only filter or display on (like `category`) are kept as ordinary columns instead of being folded into the embedding.

In [14]:
doc_texts = [f"{d['title']}. {d['content']}" for d in DOCS]
doc_vectors = np.array([_unit(v) for v in embedder.embed(doc_texts)], dtype=np.float32)
dim = int(doc_vectors.shape[1])
print(f"Embedded {len(DOCS)} docs at dimension {dim}.")

Embedded 12 docs at dimension 768.


## 2.4 Create the Table

The table stores the document plus a native `VECTOR(dim, FLOAT32)` column. The DDL drops any prior copy first so the cell is safe to re-run.

> ⚠️ **The `VECTOR` dimension must match your embedding model.** The column below is declared `VECTOR(dim, FLOAT32)`, where `dim` is the value we read from the model in §2.3 (`768` for `nomic-embed-text-v1.5`). The number in the column definition has to equal the length of the vectors you insert — declare `VECTOR(512, ...)` while the model emits 768-dim vectors and **every insert fails** with a dimension-mismatch error. We thread `dim` through programmatically precisely so the table always matches whatever model produced the embeddings.

In [15]:
import array

table_ddl = f"""
BEGIN EXECUTE IMMEDIATE 'DROP TABLE acme_docs CASCADE CONSTRAINTS PURGE';
EXCEPTION WHEN OTHERS THEN IF SQLCODE != -942 THEN RAISE; END IF; END;
/
CREATE TABLE acme_docs (
    doc_id    VARCHAR2(64) PRIMARY KEY,
    title     VARCHAR2(400),
    category  VARCHAR2(64),
    content   VARCHAR2(4000),
    embedding VECTOR({dim}, FLOAT32)
)
"""
with conn.cursor() as cur:
    for stmt in table_ddl.split("/"):
        if stmt.strip():
            cur.execute(stmt)
conn.commit()
print(f"Table ACME_DOCS created (VECTOR dim={dim}).")

Table ACME_DOCS created (VECTOR dim=768).


### Add the indexes

Two indexes power the retrieval techniques that follow:

| Index | Type | Powers |
|---|---|---|
| `acme_text_idx` | Oracle Text (`CTXSYS.CONTEXT`) | keyword & hybrid search (`CONTAINS`) |
| `acme_vec_idx` | Vector (HNSW) | semantic search + attribute filtering (the `VECTOR_INDEX_TRANSFORM` hint in §2.6.3 needs an HNSW index) |

#### What is HNSW?

**HNSW — Hierarchical Navigable Small World** — is the graph-based vector index that makes similarity search fast. Instead of comparing your query against *every* row (an exact, `O(n)` full scan), HNSW pre-builds a multi-layer graph in which each vector is a node linked to a handful of its nearest neighbours:

- **Upper layers are sparse** — few nodes, long-range links, used for big jumps across the space.
- **Lower layers are dense** — many nodes, short links, used for fine-grained local search.

A query enters at the top layer, greedily hops to whichever neighbour sits closest to it, then drops down a layer and repeats — zooming in until it lands among the true nearest neighbours in the bottom layer. The payoff is **approximate** nearest-neighbour search in roughly `O(log n)` time instead of a linear scan. "Approximate" because it can occasionally miss a true neighbour — that's the price of the speed-up, and it's tunable (see `ef` below).

#### The SQL that builds it

The statement that creates the index in the next cell is:

```sql
CREATE VECTOR INDEX acme_vec_idx ON acme_docs(embedding)
  ORGANIZATION INMEMORY NEIGHBOR GRAPH        -- build an HNSW graph in the in-memory vector pool
  DISTANCE COSINE WITH TARGET ACCURACY 90     -- distance metric + recall target (drives ef at query time)
  PARAMETERS (TYPE HNSW, NEIGHBORS 16, EFCONSTRUCTION 200);
```

`ORGANIZATION INMEMORY NEIGHBOR GRAPH` is Oracle's way of saying "store an HNSW graph in the vector memory pool." Everything that tunes the graph lives in the `PARAMETERS (...)` clause and in `TARGET ACCURACY`.

#### The two knobs: `m` and `ef`

These are the classic HNSW parameters; Oracle spells them with its own keywords:

| HNSW concept | Oracle keyword | Applies at | Controls |
|---|---|---|---|
| **`m`** — max connections per node | `NEIGHBORS` | build time | how *dense* the graph is (edges kept per node, per layer) |
| **`efConstruction`** | `EFCONSTRUCTION` | build time | how many candidates the builder weighs when wiring each node |
| **`efSearch`** | derived from `TARGET ACCURACY` | every query | how many candidates the traversal keeps "in play" while searching |

**`m` (`NEIGHBORS 16`) — graph density.** More neighbours per node ⇒ more paths through the graph ⇒ higher recall and a more robust search, but **more memory** and **slower builds/inserts**. Typical values are 16–64: too low and the graph fragments and misses neighbours; too high and you pay memory for diminishing returns.

**`efConstruction` (`200`) — build-time effort.** A larger candidate list *while building* yields a higher-quality graph (a better recall ceiling) at the cost of a slower one-time build. It does not affect query speed.

**`efSearch` — query-time effort.** This is the dial you turn *per query*: a larger `ef` explores more of the graph, pushing recall toward exact results but making each search slower. We don't set it directly — `WITH TARGET ACCURACY 90` asks Oracle to pick an `efSearch` that returns about 90% of the true top-k, and the `FETCH APPROX … WITH TARGET ACCURACY 90` clause in §2.6.3 requests that approximate path at query time. Lower the target for speed; raise it for precision.

> 🎛️ **Rule of thumb:** `m` and `efConstruction` are set when you *build* the index and trade memory & build time for the recall *ceiling*; `efSearch` is tuned when you *query* and trades latency for how close to that ceiling each search gets.

> 💡 **Key Term — HNSW index:** A graph-based vector index that turns similarity search from a full scan into a sub-second lookup. On our tiny table an exact scan is already instant, so the vector index is mainly here to (a) demonstrate the pattern and (b) enable the pre/in/post-filter hints. We tolerate it being skipped if the database's vector pool isn't configured.

![The HNSW vector index](../images/ff2_hnsw.png)


In [16]:
with conn.cursor() as cur:
    # Full-text index — REQUIRED for keyword (CONTAINS) and hybrid search.
    try:
        cur.execute("DROP INDEX acme_text_idx")
    except oracledb.DatabaseError:
        pass
    cur.execute("""
        CREATE INDEX acme_text_idx ON acme_docs(content)
        INDEXTYPE IS CTXSYS.CONTEXT PARAMETERS ('SYNC (ON COMMIT)')
    """)
    print("Oracle Text index created (keyword search ready).")

    # HNSW vector index — enables APPROX search and the pre/in/post-filter hints.
    try:
        cur.execute("DROP INDEX acme_vec_idx")
    except oracledb.DatabaseError:
        pass
    try:
        cur.execute("""
            CREATE VECTOR INDEX acme_vec_idx ON acme_docs(embedding)
            ORGANIZATION INMEMORY NEIGHBOR GRAPH
            DISTANCE COSINE WITH TARGET ACCURACY 90
            PARAMETERS (TYPE HNSW, NEIGHBORS 16, EFCONSTRUCTION 200)
        """)
        print("HNSW vector index created.")
    except oracledb.DatabaseError as e:
        print("HNSW index skipped (exact search still works):", str(e).splitlines()[0][:90])
conn.commit()

Oracle Text index created (keyword search ready).
HNSW vector index created.


## 2.5 Ingest the Documents

Each embedding is sent as a Python `array('f', ...)`, which Oracle's driver maps straight onto the `VECTOR` column. `executemany` inserts the whole batch in one round trip.

In [17]:
rows = [
    (d["doc_id"], d["title"], d["category"], d["content"], array.array("f", vec))
    for d, vec in zip(DOCS, doc_vectors.astype(np.float32).tolist())
]

with conn.cursor() as cur:
    cur.executemany(
        "INSERT INTO acme_docs (doc_id, title, category, content, embedding) "
        "VALUES (:1, :2, :3, :4, :5)",
        rows,
    )
conn.commit()

After inserting, we run a quick `SELECT COUNT(*)` as a **sanity check** — confirming all twelve rows actually landed in `ACME_DOCS` before we start querying. It is a cheap guard against a partial or failed ingest, so a later "no results" surprise is never just missing data.

In [18]:

with conn.cursor() as cur:
    cur.execute("SELECT COUNT(*) FROM acme_docs")
    print("Rows in ACME_DOCS:", cur.fetchone()[0])

Rows in ACME_DOCS: 12


## 2.6 Retrieval Techniques

> 💡 **Key Insight — No single retrieval strategy wins everywhere.** Keyword search nails exact terms but misses synonyms; vector search captures meaning but can drift off-topic; attribute filtering narrows by structured metadata; hybrid methods combine keyword + vector; graph retrieval follows relationships. We'll work through each on the same data.

First, three small helpers shared by the techniques below.

![Retrieval approaches compared](../images/ff2_retreival_approach.png)


The next cell defines four small helpers used by every technique that follows:

| Helper | What it does |
|---|---|
| `_query_vec(text)` | Embeds a query string into a unit vector (fastembed adds nomic's query-side prefix automatically). |
| `embed_query(text)` | Wraps `_query_vec` and returns the vector as a Python `array('f')` — the form Oracle's driver binds to a `:q` `VECTOR` parameter. |
| `contains_arg(phrase)` | Makes a phrase safe for Oracle Text `CONTAINS` by quoting multi-word phrases. |
| `show(rows, columns)` | Renders a `(rows, columns)` query result as a pandas DataFrame for readable output. |

In [19]:
import pandas as pd


def _query_vec(text: str):
    """Embed a query as a unit vector (fastembed applies nomic's query prefix internally)."""
    return _unit(next(embedder.query_embed(text)))


def embed_query(text: str):
    """A query vector in Oracle's expected array form (FLOAT32)."""
    return array.array("f", _query_vec(text).astype(np.float32).tolist())


def contains_arg(phrase: str) -> str:
    """Make a phrase safe for Oracle Text CONTAINS (wrap multi-word phrases in quotes)."""
    return f'"{phrase}"' if " " in phrase.strip() else phrase


def show(rows, columns):
    """Render result rows as a DataFrame for easy reading."""
    return pd.DataFrame(rows, columns=columns)

### 2.6.1 Keyword Search (Oracle Text)

Full-text search over the `content` column. `SCORE(1)` ranks by term relevance. Great for exact terms — but it only matches words that literally appear.

**How keyword search works in Oracle AI Database.** The `CONTAINS(content, ...)` predicate is powered by the **Oracle Text** index we built (`CTXSYS.CONTEXT`). At index time Oracle *tokenizes* every document — splitting text into words, lower-casing them, and recording which documents each term appears in (an **inverted index**). At query time it looks the search term up in that index instead of scanning every row, so full-text search stays fast even over large tables.

**What `SCORE(1)` means.** When a row matches, Oracle Text assigns it a **relevance score** (the `1` ties the score to this particular `CONTAINS` call). The score is higher when the term is more prominent in a document — driven by how often it occurs relative to the document's length — and is returned on a **0–100** scale. Ordering by `SCORE(1) DESC` therefore puts the most on-topic documents first.

In [20]:
def keyword_search(query: str, k: int = 5):
    sql = f"""
        SELECT doc_id, title, category, SCORE(1) AS score
        FROM acme_docs
        WHERE CONTAINS(content, :kw, 1) > 0
        ORDER BY SCORE(1) DESC
        FETCH FIRST {int(k)} ROWS ONLY
    """
    with conn.cursor() as cur:
        cur.execute(sql, kw=contains_arg(query))
        return cur.fetchall(), [d[0] for d in cur.description]

In [21]:
rows, cols = keyword_search("SSO")
show(rows, cols)

,DOC_ID,TITLE,CATEGORY,SCORE
0,sso,Single Sign-On (SSO),security,11
1,plans,Plans & Pricing,billing,5


### 2.6.2 Vector Search (semantic)

Embed the query, then rank rows by cosine similarity with `VECTOR_DISTANCE`. This matches *meaning*, so a paraphrase with none of the document's exact words still finds the right doc.

> 💡 **Teachable moment:** The query below contains none of the words "rate", "limit", or "API" — yet vector search returns the **API Rate Limits** doc first. That semantic recall is what keyword search structurally cannot do.

![Vector search](../images/ff2_vector_search.png)


**How vector search works.** Three steps: 

**(1)** embed the query with the *same* model used for the documents, so query and documents share one vector space; 

**(2)** compute the **cosine distance** between the query vector and every stored `embedding` with `VECTOR_DISTANCE(embedding, :q, COSINE)`; 

**(3)** return the closest rows. Because "closeness" in that shared space means "similar in meaning," a paraphrase with none of the document's exact words still lands near the right document.

**What the score means.** `VECTOR_DISTANCE(..., COSINE)` returns a *distance* (0 = identical direction, larger = further apart). We report `1 - distance` as a **similarity** — for our unit-length embeddings that lands in roughly `[0, 1]`, where **higher = more similar** — which is why we `ORDER BY similarity DESC`. On a large table you'd add an HNSW index and use `FETCH APPROX` to make this an *approximate* nearest-neighbour search (sub-linear) instead of scanning every row.

In [22]:
def vector_search(query: str, k: int = 5):
    q = embed_query(query)
    sql = f"""
        SELECT doc_id, title, category,
               ROUND(1 - VECTOR_DISTANCE(embedding, :q, COSINE), 4) AS similarity
        FROM acme_docs
        ORDER BY similarity DESC
        FETCH FIRST {int(k)} ROWS ONLY
    """
    with conn.cursor() as cur:
        cur.execute(sql, q=q)
        return cur.fetchall(), [d[0] for d in cur.description]


In [23]:
rows, cols = vector_search("how many requests can I send before being throttled?")
show(rows, cols)

,DOC_ID,TITLE,CATEGORY,SIMILARITY
0,rate_limits,API Rate Limits,api,0.6320
1,webhooks,Webhooks,integrations,0.5756
2,support,Support Channels & Response Times,support,0.5268
3,sla,Service Level Agreement,reliability,0.4910
4,upgrade,Upgrading Your Plan,billing,0.4897


### 2.6.3 Attribute Filtering — Pre / In / Post

Real queries usually combine semantic similarity with a **structured filter** ("only `data` docs", "only this tenant", "only last 30 days"). Oracle lets you control *when* that `WHERE` predicate is applied relative to the HNSW graph traversal, via the `VECTOR_INDEX_TRANSFORM` optimizer hint:

| Mode | Hint | What happens | Trade-off |
|---|---|---|---|
| **Pre-filter** | `pre_filter_with_join_back` | Apply the `WHERE` filter **first**, then rank the surviving rows by vector distance | Guarantees you get *topN within the filtered set*; can be costly if the filter is broad |
| **In-filter** | `in_filter_with_join_back` | The filter is checked **during** the graph traversal — only matching nodes are explored | Often the best balance; available on HNSW indexes |
| **Post-filter** | `post_filter_without_join_back` | Rank by vector distance **first**, then drop rows that fail the filter | Fastest, but can **under-return** — if most top-K vectors fail the filter, you get fewer than `k` rows |

> 💡 **Key Term — pre / in / post filtering:** the same query, the same answer set *in principle* — but the order of "filter" vs "search" changes both performance and recall. Omit the hint and Oracle's optimizer picks for you.

The `filtered_search` function below runs a vector search **restricted to one `category`**, and lets us choose the filtering strategy. Three pieces of its SQL are worth reading closely:

- **`/*+ vector_index_transform(acme_docs acme_vec_idx <filtertype>) */`** — an Oracle *optimizer hint*. It tells the planner to use our HNSW index `acme_vec_idx` on `acme_docs` and to apply the `WHERE category = :cat` predicate using the chosen strategy (`pre_filter_with_join_back`, `in_filter_with_join_back`, or `post_filter_without_join_back`). Hints are advisory — remove it and Oracle picks a strategy itself.
- **`VECTOR_DISTANCE(embedding, :q, COSINE)`** — computes the cosine distance between each row's stored `embedding` and the bound query vector `:q`. We `ORDER BY` it (closest first) and also report `1 - distance` as a similarity.
- **`FETCH APPROX FIRST k ROWS ONLY WITH TARGET ACCURACY 90`** — `APPROX` requests *approximate* nearest-neighbour search through the HNSW index instead of an exact full scan, and `TARGET ACCURACY 90` says "aim to return about 90% of the true top-k." That accuracy figure is the classic ANN trade-off: lower is faster and cheaper, higher is slower but closer to exact.

The cell defines the function and runs it once with the default pre-filter; the **next** cell then compares all three strategies side by side.

In [24]:
def filtered_search(query: str, category: str, k: int = 5,
                    filtertype: str = "pre_filter_with_join_back"):
    """Vector search restricted to a category, with the chosen pre/in/post filter strategy."""

    q = embed_query(query)

    sql = f"""
        SELECT /*+ vector_index_transform(acme_docs acme_vec_idx {filtertype}) */
               doc_id, title, category,
               ROUND(1 - VECTOR_DISTANCE(embedding, :q, COSINE), 4) AS similarity
        FROM acme_docs
        WHERE category = :cat
        ORDER BY VECTOR_DISTANCE(embedding, :q, COSINE)
        FETCH APPROX FIRST {int(k)} ROWS ONLY WITH TARGET ACCURACY 90
    """

    with conn.cursor() as cur:
        cur.execute(sql, q=q, cat=category)
        return cur.fetchall(), [d[0] for d in cur.description]

In [25]:

rows, cols = filtered_search("keep my data safe and recoverable", category="data")
show(rows, cols)

,DOC_ID,TITLE,CATEGORY,SIMILARITY
0,backups,Backups & Recovery,data,0.6402
1,export,Exporting Your Data,data,0.5365
2,regions,Data Residency & Regions,data,0.5288


**Compare the strategies by *result*, not just mechanism.** Two facts drive what you'll see below:

- **Pre-filter and in-filter return the same rows.** Both apply the `category` filter as part of ranking, so both always fill `k` from inside the category (whenever enough matching docs exist). They differ in *how* — and *how fast* — Oracle gets there, not in the rows.
- **Post-filter can return *fewer* rows.** It ranks across *all* categories first, then drops whatever fails the filter — so if the query's global top-k aren't in your category, you get back fewer than `k` (sometimes zero).

To actually *see* them diverge we run two cases: one where the filter **aligns** with the query (post-filter keeps its hits) and one where it's **misaligned** (post-filter collapses). We compute the post-filter step explicitly — rank globally, then filter — because on a 12-row table Oracle's optimizer falls back to an exact scan that would otherwise mask the effect.

In [26]:
def compare_filters(query, category, k):
    """Show how pre-/in-/post-filtering diverge for one query + filter + k."""
    print(f"query={query!r}   filter: category={category!r}   k={k}")

    # Pre- and in-filter apply the category filter as part of ranking, so both
    # always fill k from inside the category (when enough matching docs exist).
    for ft in ("pre_filter_with_join_back", "in_filter_with_join_back"):
        rows, _ = filtered_search(query, category, k=k, filtertype=ft)
        print(f"  {ft:30s} -> {len(rows)} rows  {[r[0] for r in rows]}")

    # Post-filter, made explicit: rank ALL docs by vector first, then drop the
    # ones that fail the filter. (Oracle's post_filter_without_join_back does this
    # under the hood; on a 12-row table the optimizer falls back to an exact scan
    # that would hide the effect, so we show the mechanism directly.)
    all_ranked, _ = vector_search(query, k=k)              # top-k across every category
    survivors = [r for r in all_ranked if r[2] == category]
    print(f"  {'post_filter_without_join_back':30s} -> {len(survivors)} rows  {[r[0] for r in survivors]}")
    print(f"     global top-{k}: {[(r[0], r[2]) for r in all_ranked]}")
    print(f"     -> only the {category!r} rows survive\n")


print("① Filter ALIGNED with the query (top hits are already 'data') -> post-filter keeps them:")
compare_filters("keep my data safe and recoverable", category="data", k=3)

print("\n② Filter MISALIGNED with the query (top hits are billing) -> post-filter under-returns:")
compare_filters("how do I pay my invoice and change my plan", category="data", k=2)

① Filter ALIGNED with the query (top hits are already 'data') -> post-filter keeps them:
query='keep my data safe and recoverable'   filter: category='data'   k=3
  pre_filter_with_join_back      -> 3 rows  ['backups', 'export', 'regions']
  in_filter_with_join_back       -> 3 rows  ['backups', 'export', 'regions']
  post_filter_without_join_back  -> 3 rows  ['backups', 'export', 'regions']
     global top-3: [('backups', 'data'), ('export', 'data'), ('regions', 'data')]
     -> only the 'data' rows survive


② Filter MISALIGNED with the query (top hits are billing) -> post-filter under-returns:
query='how do I pay my invoice and change my plan'   filter: category='data'   k=2
  pre_filter_with_join_back      -> 2 rows  ['backups', 'regions']
  in_filter_with_join_back       -> 2 rows  ['backups', 'regions']
  post_filter_without_join_back  -> 0 rows  []
     global top-2: [('upgrade', 'billing'), ('plans', 'billing')]
     -> only the 'data' rows survive



> 💡 **Teachable moment — why post-filter can under-return:** Imagine 1M docs and you ask for the top 10 by vector, then keep only `category='data'`. If none of the global top 10 happen to be `data`, you get **zero** rows back. Pre-filter and in-filter avoid that by constraining to `data` *before/while* ranking — so they always fill `k` if enough matches exist. That's the practical reason to prefer pre/in-filter when the filter is selective.

### 2.6.4 Hybrid Search — Keyword + Vector (RRF)

> 💡 **Key Term — RRF (Reciprocal Rank Fusion):** Merge two independent ranked lists — one from vector search, one from keyword search — using each item's *rank position*, not its raw score: `score = 1/(k + rank_vector) + 1/(k + rank_keyword)`. It normalizes across incompatible scoring scales and needs no weight tuning.

![Hybrid search — Reciprocal Rank Fusion](../images/ff2_rrf.png)


**How is this different from the filtering we just did?** Attribute filtering (§2.6.3) works from *one* signal — vector similarity — and uses a structured `WHERE` predicate to decide *which rows are even eligible* (e.g. only `category='data'`). It **narrows** a single search.

RRF does something different: it runs **two independent searches** — keyword and vector — each producing its own ranked list, then **fuses** them. Nothing is filtered out; instead, a document that ranks well in *either* list (or modestly in both) rises to the top. So filtering is about *eligibility on an attribute*, while RRF is about *combining two retrieval signals* — catching exact-term matches and semantic matches in one result set.

In [27]:
def rrf_search(query: str, k: int = 5, per_list: int = 10, rrf_k: int = 60):
    q = embed_query(query)
    sql = f"""
        WITH
        vec AS (
            SELECT doc_id, title,
                   ROW_NUMBER() OVER (ORDER BY VECTOR_DISTANCE(embedding, :q, COSINE)) AS r_vec
            FROM acme_docs
            ORDER BY VECTOR_DISTANCE(embedding, :q, COSINE)
            FETCH FIRST {int(per_list)} ROWS ONLY
        ),
        txt AS (
            SELECT doc_id, title,
                   ROW_NUMBER() OVER (ORDER BY SCORE(1) DESC) AS r_txt
            FROM acme_docs
            WHERE CONTAINS(content, :kw, 1) > 0
            ORDER BY SCORE(1) DESC
            FETCH FIRST {int(per_list)} ROWS ONLY
        ),
        fused AS (
            SELECT COALESCE(v.doc_id, t.doc_id) AS doc_id,
                   COALESCE(v.title, t.title) AS title,
                   NVL(v.r_vec, 999999) AS r_vec,
                   NVL(t.r_txt, 999999) AS r_txt
            FROM vec v FULL OUTER JOIN txt t ON t.doc_id = v.doc_id
        )
        SELECT doc_id, title, r_vec, r_txt,
               ROUND(1.0/(:rk + r_vec) + 1.0/(:rk + r_txt), 6) AS rrf_score
        FROM fused
        ORDER BY rrf_score DESC, title
        FETCH FIRST {int(k)} ROWS ONLY
    """
    with conn.cursor() as cur:
        cur.execute(sql, q=q, kw=contains_arg(query), rk=rrf_k)
        return cur.fetchall(), [d[0] for d in cur.description]


rows, cols = rrf_search("Enterprise")
show(rows, cols)

,DOC_ID,TITLE,R_VEC,R_TXT,RRF_SCORE
0,backups,Backups & Recovery,3,1,0.032266
1,support,Support Channels & Response Times,2,3,0.032002
2,sso,Single Sign-On (SSO),1,5,0.031778
3,plans,Plans & Pricing,5,2,0.031514
4,rate_limits,API Rate Limits,4,6,0.030777


> 💡 **The native, turnkey way — Hybrid Vector Index:** Oracle 23ai/26ai can combine full-text and semantic search in a *single* index and query it with one call:
>
> ```sql
> CREATE HYBRID VECTOR INDEX acme_hybrid_idx ON acme_docs(content)
>   PARAMETERS ('MODEL my_indb_model');
>
> SELECT DBMS_HYBRID_VECTOR.SEARCH(json('{
>   "hybrid_index_name": "acme_hybrid_idx",
>   "vector": { "search_text": "rate limits" },
>   "text":   { "contains": "Pro" },
>   "return": { "topN": 5 } }')) FROM dual;
> ```
>
> It requires an **in-database ONNX embedding model** (it chunks and embeds the documents itself), so it's the production path for large document corpora. Here we use client-side embeddings and explicit SQL so each retrieval step is visible, fusing keyword + vector with RRF above.

### 2.6.5 Graph Retrieval (SQL Property Graph)

> 💡 **Key Term — Graph retrieval:** Seed the search with vector similarity, then **traverse relationship edges** to pull in docs the vector search alone would miss. We build two edge types:
> - `SIMILAR_TO` — each doc's nearest neighbours by embedding (computed once, stored as edges)
> - `IN_CATEGORY` — links every doc to its category vertex, so two docs in the same category are two hops apart
>
> The query uses Oracle's SQL property graph with the SQL/PGQ `GRAPH_TABLE ... MATCH` operator — the current standard for graph queries in Oracle AI Database.
>
> **Prerequisite:** the `CREATE PROPERTY GRAPH` privilege — granted automatically by `oracle.sh` (and in prepared workshop environments).

We build this in four small steps: 

(1) create the edge tables, 

(2) compute and load the edges, 

(3) register the property graph, 

(4) query it.

![Graph retrieval (GraphRAG)](../images/ff2_graphrag.png)


**Step 1 — create the relational edge tables** that will back the graph.

In [28]:
graph_ddl = """
BEGIN EXECUTE IMMEDIATE 'DROP TABLE doc_similarities'; EXCEPTION WHEN OTHERS THEN IF SQLCODE != -942 THEN RAISE; END IF; END;
/
BEGIN EXECUTE IMMEDIATE 'DROP TABLE doc_categories'; EXCEPTION WHEN OTHERS THEN IF SQLCODE != -942 THEN RAISE; END IF; END;
/
BEGIN EXECUTE IMMEDIATE 'DROP TABLE categories'; EXCEPTION WHEN OTHERS THEN IF SQLCODE != -942 THEN RAISE; END IF; END;
/
CREATE TABLE categories (category VARCHAR2(64) PRIMARY KEY)
/
CREATE TABLE doc_categories (
    doc_id   VARCHAR2(64) NOT NULL,
    category VARCHAR2(64) NOT NULL,
    CONSTRAINT pk_doc_categories PRIMARY KEY (doc_id, category),
    CONSTRAINT fk_dc_doc FOREIGN KEY (doc_id) REFERENCES acme_docs(doc_id),
    CONSTRAINT fk_dc_cat FOREIGN KEY (category) REFERENCES categories(category)
)
/
CREATE TABLE doc_similarities (
    source_doc_id VARCHAR2(64) NOT NULL,
    target_doc_id VARCHAR2(64) NOT NULL,
    sim_score NUMBER(8,6) NOT NULL,
    rank_no   NUMBER(5) NOT NULL,
    CONSTRAINT pk_doc_similarities PRIMARY KEY (source_doc_id, target_doc_id),
    CONSTRAINT fk_ds_src FOREIGN KEY (source_doc_id) REFERENCES acme_docs(doc_id),
    CONSTRAINT fk_ds_tgt FOREIGN KEY (target_doc_id) REFERENCES acme_docs(doc_id),
    CONSTRAINT ck_ds_not_self CHECK (source_doc_id <> target_doc_id)
)
"""
with conn.cursor() as cur:
    for stmt in graph_ddl.split("/"):
        if stmt.strip():
            cur.execute(stmt)
conn.commit()
print("Edge tables created: categories, doc_categories, doc_similarities.")

Edge tables created: categories, doc_categories, doc_similarities.


**Step 2 — compute and load the edges.** Category edges come straight from each doc's `category`; similarity edges are each doc's top-3 nearest neighbours by cosine, computed once in NumPy.

In [29]:
doc_ids = [d["doc_id"] for d in DOCS]
categories = sorted({d["category"] for d in DOCS})
doc_cat_rows = [(d["doc_id"], d["category"]) for d in DOCS]

sim = doc_vectors @ doc_vectors.T          # cosine similarity (vectors are normalized)
np.fill_diagonal(sim, -np.inf)             # never link a doc to itself
TOP_NEIGHBORS = 3
sim_rows = []
for i, src in enumerate(doc_ids):
    for rank, j in enumerate(np.argsort(sim[i])[::-1][:TOP_NEIGHBORS], start=1):
        sim_rows.append((src, doc_ids[j], round(float(sim[i][j]), 6), rank))

with conn.cursor() as cur:
    cur.executemany("INSERT INTO categories (category) VALUES (:1)", [(c,) for c in categories])
    cur.executemany("INSERT INTO doc_categories (doc_id, category) VALUES (:1, :2)", doc_cat_rows)
    cur.executemany(
        "INSERT INTO doc_similarities (source_doc_id, target_doc_id, sim_score, rank_no) "
        "VALUES (:1, :2, :3, :4)",
        sim_rows,
    )
conn.commit()
print(f"Loaded {len(categories)} categories, {len(doc_cat_rows)} IN_CATEGORY edges, "
      f"{len(sim_rows)} SIMILAR_TO edges.")

Loaded 8 categories, 12 IN_CATEGORY edges, 36 SIMILAR_TO edges.


**Step 3 — register the SQL property graph** over the docs (vertices) and the two edge tables.

In [30]:
with conn.cursor() as cur:
    cur.execute("SELECT COUNT(*) FROM user_property_graphs WHERE graph_name = 'ACME_GRAPH'")
    if cur.fetchone()[0] > 0:
        cur.execute("DROP PROPERTY GRAPH acme_graph")
    cur.execute("""
        CREATE PROPERTY GRAPH acme_graph
        VERTEX TABLES (
            acme_docs  KEY (doc_id)   LABEL doc      PROPERTIES (doc_id, title, category),
            categories KEY (category) LABEL category PROPERTIES (category)
        )
        EDGE TABLES (
            doc_categories KEY (doc_id, category)
                SOURCE KEY (doc_id) REFERENCES acme_docs (doc_id)
                DESTINATION KEY (category) REFERENCES categories (category)
                LABEL in_category,
            doc_similarities KEY (source_doc_id, target_doc_id)
                SOURCE KEY (source_doc_id) REFERENCES acme_docs (doc_id)
                DESTINATION KEY (target_doc_id) REFERENCES acme_docs (doc_id)
                LABEL similar_to PROPERTIES (sim_score, rank_no)
        )
    """)
conn.commit()
print("Property graph ACME_GRAPH created.")

Property graph ACME_GRAPH created.


**Step 4 — query it.** The SQL below has four CTEs that mirror the algorithm:
1. `seed` — top docs by vector similarity (the starting points)
2. `sim_hops` — `GRAPH_TABLE … MATCH` across `SIMILAR_TO` edges from each seed
3. `cat_hops` — two-hop traversal `doc → category → doc` (same-category docs)
4. `scored` — blend the seed score with the edge contribution (similar edges weighted higher than same-category)

In [31]:
def graph_search(query: str, k: int = 5, seed_k: int = 5):
    """Vector-seed, then expand across SIMILAR_TO and same-category edges; blend the scores."""
    seed_k = max(seed_k, k)
    q = embed_query(query)
    sql = f"""
        WITH seed AS (
            SELECT doc_id, 1 - VECTOR_DISTANCE(embedding, :q, COSINE) AS seed_score
            FROM acme_docs
            ORDER BY seed_score DESC
            FETCH FIRST {int(seed_k)} ROWS ONLY
        ),
        seed_hits AS (
            SELECT doc_id AS candidate, seed_score, 'seed' AS rel, seed_score AS edge_score
            FROM seed
        ),
        sim_hops AS (
            SELECT gt.target_doc_id AS candidate, s.seed_score, 'similar_to' AS rel, gt.edge_score
            FROM seed s
            JOIN GRAPH_TABLE(acme_graph
                MATCH (src IS doc)-[e IS similar_to]->(dst IS doc)
                COLUMNS (src.doc_id AS source_doc_id, dst.doc_id AS target_doc_id, e.sim_score AS edge_score)
            ) gt ON gt.source_doc_id = s.doc_id
        ),
        cat_hops AS (
            SELECT gt.target_doc_id AS candidate, s.seed_score, 'same_category' AS rel, 1.0 AS edge_score
            FROM seed s
            JOIN GRAPH_TABLE(acme_graph
                MATCH (src IS doc)-[IS in_category]->(c IS category)<-[IS in_category]-(dst IS doc)
                COLUMNS (src.doc_id AS source_doc_id, dst.doc_id AS target_doc_id)
            ) gt ON gt.source_doc_id = s.doc_id
            WHERE gt.target_doc_id <> s.doc_id
        ),
        candidates AS (
            SELECT * FROM seed_hits
            UNION ALL SELECT * FROM sim_hops
            UNION ALL SELECT * FROM cat_hops
        ),
        scored AS (
            SELECT candidate AS doc_id,
                   MAX(CASE rel
                       WHEN 'seed'          THEN seed_score
                       WHEN 'similar_to'    THEN 0.70 * seed_score + 0.30 * edge_score
                       WHEN 'same_category' THEN 0.85 * seed_score + 0.15 * edge_score
                       ELSE seed_score END) AS graph_score
            FROM candidates
            GROUP BY candidate
        )
        SELECT d.doc_id, d.title, d.category, ROUND(sc.graph_score, 4) AS graph_score
        FROM scored sc JOIN acme_docs d ON d.doc_id = sc.doc_id
        ORDER BY graph_score DESC
        FETCH FIRST {int(k)} ROWS ONLY
    """
    with conn.cursor() as cur:
        cur.execute(sql, q=q)
        return cur.fetchall(), [c[0] for c in cur.description]

In [32]:
# Seeds on the SSO doc, then expands to its similar docs and the rest of the security/related set:
rows, cols = graph_search("How do I sign in with my company identity provider?")
show(rows, cols)

,DOC_ID,TITLE,CATEGORY,GRAPH_SCORE
0,sso,Single Sign-On (SSO),security,0.6964
1,sla,Service Level Agreement,reliability,0.6709
2,support,Support Channels & Response Times,support,0.6652
3,roles,Team Roles & Permissions,account,0.6623
4,plans,Plans & Pricing,billing,0.6126


### 2.6.6 Compare the Techniques

Run the query-driven techniques on the same query and line up their top results. Keyword returns only literal matches; vector, hybrid, and graph surface semantically or structurally related docs too.

In [33]:
techniques = {
    "keyword": keyword_search,
    "vector": vector_search,
    "hybrid (rrf)": rrf_search,
    "graph": graph_search,
}

query = "webhooks"

In [34]:
print(f"Query: {query!r}\n")

for name, fn in techniques.items():
    rows, cols = fn(query, k=3)
    titles = [row[cols.index("TITLE")] for row in rows]
    print(f"{name:>13}: {titles}")

Query: 'webhooks'

      keyword: ['Webhooks']
       vector: ['Webhooks', 'Creating & Rotating API Keys', 'Exporting Your Data']
 hybrid (rrf): ['Webhooks', 'Creating & Rotating API Keys', 'Exporting Your Data']
        graph: ['Backups & Recovery', 'Webhooks', 'API Rate Limits']


## 2.7 Augment the Prompt, Then Generate

Now the RAG payoff. We wrap vector search in a single `retrieve()` helper — the **one retrieval interface the rest of the notebook reuses** (Form Factors 3 and 4 import it, so retrieval improvements propagate everywhere).

In [ ]:
def retrieve(query: str, k: int = 3):
    """Unified retriever (Oracle vector search). Returns [(content, similarity), ...]."""
    q = embed_query(query)
    sql = f"""
        SELECT content, ROUND(1 - VECTOR_DISTANCE(embedding, :q, COSINE), 4) AS similarity
        FROM acme_docs
        ORDER BY similarity DESC
        FETCH FIRST {int(k)} ROWS ONLY
    """
    with conn.cursor() as cur:
        cur.execute(sql, q=q)
        return [(content, float(sim)) for content, sim in cur.fetchall()]

0.755  Acme Cloud enforces API rate limits per plan. The Free plan ...
0.619  Acme Cloud offers three plans. Free includes 1 project and c...
0.615  Acme Cloud guarantees 99.9% uptime on the Pro plan and 99.99...


In [ ]:
for content, score in retrieve("Pro plan rate limit", k=3):
    print(f"{score:.3f}  {content[:60]}...")

Then we feed the retrieved context to Claude, instructing it to answer **only** from that context and to cite it with bracketed numbers.

In [36]:
def rag_answer(query: str, k: int = 3) -> str:
    """Form Factor 2: retrieve from Oracle, then generate a grounded, cited answer."""
    hits = retrieve(query, k)
    context = "\n".join(f"[{i + 1}] {doc}" for i, (doc, _) in enumerate(hits))

    system = (
        "You are the Acme Cloud support assistant. Answer the question using ONLY the "
        "context below. Cite the sources you use with bracketed numbers like [1]. "
        "If the answer is not in the context, say you don't have that information.\n\n"
        f"Context:\n{context}"
    )
    response = client.messages.create(
        model=MODEL,
        max_tokens=MAX_TOKENS,
        system=system,
        messages=[{"role": "user", "content": query}],
    )
    return text_of(response)


# The same question that stumped the bare chatbot — now grounded in Oracle-backed docs:
print(rag_answer("What is the exact API rate limit, in requests per minute, on Acme Cloud's Pro plan?"))

The Pro plan allows 1,000 requests per minute [1].


> ### 📌 Key Takeaways — Form Factor 2
> - **RAG = retrieve, then generate.** The model answers from *your* data placed in the prompt, with citations — fixing the chatbot's data gap.
> - **Oracle AI Database** stores text, vectors, and graphs in one engine, so every retrieval technique is just SQL against one table.
> - **No retrieval strategy wins everywhere:** keyword (exact), vector (meaning), **attribute filtering** (pre/in/post — order of filter vs. search), **hybrid** (RRF, or the native `DBMS_HYBRID_VECTOR.SEARCH`), and **graph** (relationships).

# Form Factor 3 — The LLM-Driven Workflow

> 💡 **Key Term — Workflow:** Several LLM calls (and ordinary code) composed into a fixed, predictable pipeline. The sequence of steps is decided by **you, in code** — the model fills in each step, but it doesn't choose what happens next.

A single RAG call is great for "answer this question." Real jobs, though, have *stages* — so here we automate an entire task end-to-end instead of answering one question.

### The use case & scenario

**Scenario — Acme Cloud's support inbox.** Customer messages arrive around the clock: billing disputes, technical problems, account questions, feature requests. Normally a human has to *read* each one, *decide* who should handle it and how urgent it is, *look up* the relevant documentation, *write* an accurate reply, and *double-check* that reply before it goes out. That work is repetitive, high-volume, and easy to get wrong at 2 a.m.

This workflow **automates that whole loop**: a raw message goes in, and a grounded, quality-checked draft reply — plus a routing/escalation decision — comes out, with no human writing each response. It's the classic "first-line support triage" automation.

### What the pipeline actually does

We wire single-purpose steps into a fixed sequence. Each step is a small, well-defined job, and *our code* — not the model — decides the order and the branching (this is exactly the order in `handle_support_message`, §3.4):

1. **Classify** *(§3.1 — LLM + structured output)* — read the message and return clean JSON: `category` (billing / technical / account / feature_request / other), `urgency` (low / medium / high), and a one-line `summary`. Forcing a JSON schema lets every later step branch on these fields with no fragile string parsing.
2. **Route** *(§3.4 — plain code)* — turn that structured result into a deterministic decision, e.g. flag **high-urgency billing** for a human to follow up. It's an ordinary `if`; no LLM needed once the message is classified.
3. **Retrieve** *(RAG, reused from Form Factor 2)* — fetch the most relevant Acme Cloud docs for the message so the reply is grounded in real documentation instead of the model's guesswork.
4. **Draft** *(§3.2 — LLM)* — write a concise, cited reply using *only* the retrieved context.
5. **Review → revise** *(§3.3 + §3.4 — evaluator–optimizer)* — a second, *independent* LLM call grades the draft, returning `approved` plus `feedback`. If it isn't approved, our code feeds the feedback back in and asks for a revision — a small quality-control loop that runs a bounded number of times.

The final output is a dict bundling the classification, the escalation flag, and the polished reply — the structured result a real ticketing system would store and act on.

> 💡 **Key Insight — Workflows make LLMs reliable:** Breaking a task into small, single-purpose steps with checks between them yields far more consistent results than one big do-everything prompt. Anthropic calls these patterns *prompt chaining*, *routing*, and *evaluator–optimizer* in [Building Effective Agents](https://www.anthropic.com/research/building-effective-agents).

![The LLM-driven workflow](../images/ff3_workflow.png)

![Anatomy of a workflow](../images/ff3_anatomy.png)

![Human in the loop](../images/ff3_human_in_the_loop.png)


## 3.1 Classify the Message (with structured output)

The first step routes the message. We use **structured outputs** (`output_config`) to force Claude to return clean JSON that our code can branch on — no fragile string parsing.

> 💡 **Teachable moment:** A JSON Schema with `additionalProperties: False` and every field `required` is what makes the output *machine-reliable*. The model is constrained to your shape, so `json.loads` always succeeds and your `if`/`for` logic can trust the keys.

In [37]:
import json

CLASSIFY_SCHEMA = {
    "type": "object",
    "properties": {
        "category": {
            "type": "string",
            "enum": ["billing", "technical", "account", "feature_request", "other"],
        },
        "urgency": {"type": "string", "enum": ["low", "medium", "high"]},
        "summary": {"type": "string", "description": "One-line summary of the request."},
    },
    "required": ["category", "urgency", "summary"],
    "additionalProperties": False,
}

In [38]:
def classify(message: str) -> dict:
    """Step 1: return {category, urgency, summary} as validated JSON."""
    response = client.messages.create(
        model=MODEL,
        max_tokens=512,
        system="Classify the incoming Acme Cloud support message.",
        messages=[{"role": "user", "content": message}],
        output_config={"format": {"type": "json_schema", "schema": CLASSIFY_SCHEMA}},
    )
    return json.loads(text_of(response))


classify("I've been double-charged for my Pro seats this month and need this fixed before our board demo tomorrow.")

{'category': 'billing',
 'urgency': 'high',
 'summary': "Customer was double-charged for Pro seats and needs an urgent fix before tomorrow's board demo."}

> 💡 **Why this matters in a production workload.** Forcing the model's output into a fixed JSON schema is what makes this first step *safe to build on*. In a real support pipeline this single classification drives a lot:
>
> - **Routing** — billing issues go to billing, technical to engineering.
> - **Prioritization & SLA** — an `urgency: "high"` result can page a human or jump the queue.
> - **Automation gates** — only well-classified, low-risk requests get an auto-reply; the rest are escalated to a person.
> - **Analytics** — consistent categories let you measure ticket volume and trends over time.
>
> Because the schema guarantees the shape (`additionalProperties: false`, every field required), the code that branches on `category` / `urgency` can trust those keys exist — no brittle string parsing, no surprise `KeyError` in the middle of the night. That reliability is much of the difference between a demo and a production workflow.

### 3.1.1 Sanity-check the classifier against human labels

Those real tickets came with **human-assigned labels** — a `queue` (which team owns it) and a `priority` (low/medium/high). That gives us a free **ground truth**: run `classify()` on a few real tickets and see how often the model agrees with the humans.

> 💡 **Teachable moment:** `urgency` lines up cleanly with the dataset's `priority` (both low/medium/high), so that's a direct agreement check. `category` is fuzzier — the dataset's queues use a *different taxonomy* than our five categories, so we map `queue → category` first. That mismatch is itself the lesson: real-world labels rarely match your schema 1:1, and any "accuracy" number depends on the mapping you choose. Scaled from a handful of tickets to hundreds, this same compare-against-labels loop is exactly how you'd build an **eval** for the classifier.

In [39]:
def load_support_tickets(n: int = 6) -> list:
    """Load a small sample of real, human-labeled support tickets from Hugging Face.

    Pulls a tiny slice of `Tobi-Bueck/customer-support-tickets` via the
    datasets-server REST API (no full download), keeping each ticket's human
    labels — `queue` (support category) and `priority` (low/medium/high). Falls
    back to an inline sample if offline, so this always runs. Reused by Form
    Factor 5 to stock its sandbox.
    """
    fallback = [
        {"queue": "Technical Support",               "priority": "high",   "subject": "Account Disruption"},
        {"queue": "Billing and Payments",            "priority": "low",    "subject": "Inquiry Regarding Invoice Details"},
        {"queue": "Product Support",                 "priority": "medium", "subject": "VPN Access Issue"},
        {"queue": "Service Outages and Maintenance", "priority": "high",   "subject": "System Interruptions"},
        {"queue": "Returns and Exchanges",           "priority": "medium", "subject": "Query About Smart Home System Integration"},
        {"queue": "Sales and Pre-Sales",             "priority": "medium", "subject": "Question About Software Compatibility"},
        {"queue": "IT Support",                      "priority": "medium", "subject": "Technical Problem with Cloud SaaS Service"},
        {"queue": "Customer Service",                "priority": "low",    "subject": "Inquiry for In-Depth Service Details"},
    ]
    try:
        import requests  # bundles CA certs (certifi) -> robust TLS across platforms
        resp = requests.get(
            "https://datasets-server.huggingface.co/rows",
            params={"dataset": "Tobi-Bueck/customer-support-tickets", "config": "default",
                    "split": "train", "offset": 0, "length": 100},  # window, then filter to English
            timeout=20,
        )
        resp.raise_for_status()
        tickets = []
        for item in resp.json()["rows"]:
            r = item["row"]
            subject = (r.get("subject") or "").strip()
            if r.get("language") != "en" or not subject:
                continue
            tickets.append({"queue": r.get("queue") or "General Inquiry",
                            "priority": r.get("priority") or "medium",
                            "subject": subject})
            if len(tickets) >= n:
                break
        return tickets or fallback[:n]
    except Exception:
        return fallback[:n]


# The dataset's `queue` is a different taxonomy than our classifier's five
# categories, so map it across before checking agreement (approximate by design).
QUEUE_TO_CATEGORY = {
    "Billing and Payments": "billing",
    "Technical Support": "technical",
    "IT Support": "technical",
    "Product Support": "technical",
    "Service Outages and Maintenance": "technical",
    "Customer Service": "account",
    "Returns and Exchanges": "other",
    "Sales and Pre-Sales": "other",
    "General Inquiry": "other",
    "Human Resources": "other",
}

rows = []
for t in load_support_tickets(6):
    pred = classify(t["subject"])  # the LLM's structured-output call from §3.1
    gold_cat = QUEUE_TO_CATEGORY.get(t["queue"], "other")
    rows.append({
        "message": t["subject"][:40],
        "gold_category": gold_cat,
        "model_category": pred["category"],
        "cat": "✓" if pred["category"] == gold_cat else "✗",
        "gold_urgency": t["priority"],
        "model_urgency": pred["urgency"],
        "urg": "✓" if pred["urgency"] == t["priority"] else "✗",
    })

df = pd.DataFrame(rows)
print(f"Category agreement: {(df['cat'] == '✓').mean():.0%}    "
      f"Urgency agreement: {(df['urg'] == '✓').mean():.0%}")
df

Category agreement: 50%    Urgency agreement: 33%


,message,gold_category,model_category,cat,gold_urgency,model_urgency,urg
0,Account Disruption,technical,account,✗,high,medium,✗
1,Query About Smart Home System Integratio,other,feature_request,✗,medium,low,✗
2,Inquiry Regarding Invoice Details,billing,billing,✓,low,low,✓
3,Question About Marketing Agency Software,other,other,✓,medium,low,✗
4,Feature Query,technical,feature_request,✗,high,low,✗
5,System Interruptions,technical,technical,✓,high,high,✓


## 3.2 Draft a Reply

Grounded in retrieved context (reusing `retrieve()` from Form Factor 2), Claude drafts a concise, cited reply.

In [40]:
def draft_reply(message: str, context: str) -> str:
    """Step 3: draft a support reply grounded in retrieved context."""
    system = (
        "You are an Acme Cloud support agent. Write a friendly, accurate reply using ONLY "
        "the provided context. Cite facts with [n]. Keep it under 120 words."
    )
    response = client.messages.create(
        model=MODEL,
        max_tokens=MAX_TOKENS,
        system=system,
        messages=[{"role": "user", "content": f"Customer message:\n{message}\n\nContext:\n{context}"}],
    )
    return text_of(response)

## 3.3 Review the Draft

A second, independent call grades the draft for accuracy and tone — again returning structured JSON, so our code can act on the verdict (approve, or send feedback for a revision).

> 💡 **Teachable moment — the evaluator–optimizer pattern:** Using a *separate* LLM call to check the first one's work is one of the most effective reliability techniques. The reviewer has a single, narrow job, so it catches mistakes the drafter is blind to.

In [41]:
REVIEW_SCHEMA = {
    "type": "object",
    "properties": {
        "approved": {"type": "boolean"},
        "feedback": {"type": "string", "description": "What to fix if not approved."},
    },
    "required": ["approved", "feedback"],
    "additionalProperties": False,
}

### Aside — the `output_config` argument (structured outputs)

`review()` doesn't just want *text* back from Claude; it needs a verdict its own code can branch on (`if not approved: revise`). The `output_config` argument is what guarantees that shape:

```python
output_config={"format": {"type": "json_schema", "schema": REVIEW_SCHEMA}}
```

This tells the Messages API to **constrain the response to valid JSON that matches `REVIEW_SCHEMA`** — every key present, correct types, nothing extra. `json.loads(text_of(response))` then always succeeds on a normal completion, so there is no defensive parsing, regex, or "repair the JSON" step. (`classify()` above uses the same mechanism; here it is what makes the approve/revise gate reliable.)

From the [Anthropic structured-outputs docs](https://platform.claude.com/docs/en/build-with-claude/structured-outputs):

- `output_config.format` is the canonical parameter — the older top-level `output_format` is deprecated.
- Every object in the schema must set `additionalProperties: false` (our `REVIEW_SCHEMA` does), and listing fields in `required` is what forbids missing or stray keys.
- The first call with a new schema pays a one-time compilation cost; identical schemas are then cached for ~24 hours.
- Supported on Claude Opus 4.8, Sonnet 4.6, Haiku 4.5, and Fable 5. It can't be combined with citations or assistant prefills, and if the model refuses (`stop_reason: "refusal"`) or hits `max_tokens`, the output may be incomplete — so treat schema-conformance as guaranteed only on a normal completion.

> 💡 **Teachable moment:** The Python SDK also offers `client.messages.parse()` with a Pydantic model, which validates and hands back a typed object for you. This notebook uses the raw `output_config` + `json.loads` form on purpose, to keep the JSON-Schema mechanics visible.

In [42]:

def review(message: str, draft: str, context: str) -> dict:
    """Step 4: QA the draft. Returns {approved: bool, feedback: str}."""
    system = (
        "You are a support QA reviewer. Approve the draft only if it is accurate per the "
        "context, on-topic, and professional. Otherwise set approved=false and say what to fix."
    )
    prompt = f"Customer message:\n{message}\n\nContext:\n{context}\n\nDraft reply:\n{draft}"
    response = client.messages.create(
        model=MODEL,
        max_tokens=512,
        system=system,
        messages=[{"role": "user", "content": prompt}],
        output_config={"format": {"type": "json_schema", "schema": REVIEW_SCHEMA}},
    )
    return json.loads(text_of(response))

## 3.4 Orchestrate the Pipeline

Now the control flow — and notice it's all **ordinary Python**: an `if`, a `for`, a gate. *Your code* decides the order of operations (classify → route → retrieve → draft → review → maybe revise); each LLM call does one well-defined job.

In [43]:
def handle_support_message(message: str, max_revisions: int = 1) -> dict:
    # Step 1 — classify (LLM)
    info = classify(message)
    print(f"① classified: {info['category']} / urgency={info['urgency']}")

    # Step 2 — route (your code). High-urgency billing gets flagged for a human.
    escalate = info["category"] == "billing" and info["urgency"] == "high"
    if escalate:
        print("② routing: flagged for human follow-up (urgent billing)")

    # Step 3 — retrieve (RAG, reused from Form Factor 2)
    hits = retrieve(message, k=3)
    context = "\n".join(f"[{i + 1}] {doc}" for i, (doc, _) in enumerate(hits))
    print(f"③ retrieved {len(hits)} docs")

    # Step 4 — draft (LLM)
    draft = draft_reply(message, context)

    # Step 5 — review-and-revise gate (your code controls the loop)
    for attempt in range(max_revisions + 1):
        verdict = review(message, draft, context)
        print(f"④ review attempt {attempt + 1}: approved={verdict['approved']}")
        if verdict["approved"]:
            break
        print(f"   ↳ revising: {verdict['feedback']}")
        draft = draft_reply(
            message,
            context + f"\n\nReviewer feedback to address: {verdict['feedback']}",
        )

    return {"classification": info, "escalated": escalate, "reply": draft}


result = handle_support_message(
    "I've been double-charged for my Pro seats this month and need this fixed before our board demo tomorrow."
)
print("\n--- Final reply ---\n" + result["reply"])

① classified: billing / urgency=high
② routing: flagged for human follow-up (urgent billing)
③ retrieved 3 docs
④ review attempt 1: approved=True

--- Final reply ---
Hi there,

I'm sorry for the trouble, especially with your board demo tomorrow. I want to get this sorted for you.

To investigate the duplicate charge, our billing team will need to review your account directly, as I don't have details on the specific transaction here. Since you're on the Pro plan, email support responds within one business day [1], and I'll flag your case so we can prioritize it ahead of your demo.

Could you reply with your account email and the two charge dates/amounts? That will help us resolve and refund the duplicate as quickly as possible.

Thanks for your patience—we'll make this right.


> 💡 **Key Insight — Who's in charge?** In a workflow, *the pipeline is fixed*: classify, then retrieve, then draft, then review — always in that order. That predictability is a feature. But it also means the system can't handle anything you didn't wire up in advance. When you want the system to *decide its own steps*, you've reached the next rung: the agent.
>
> For genuinely hard reasoning inside a step, enable [adaptive thinking](https://platform.claude.com/docs/en/build-with-claude/adaptive-thinking) by adding `thinking={"type": "adaptive"}` to any `messages.create` call.

> ### 📌 Key Takeaways — Form Factor 3
> - A **workflow** is multiple LLM calls wired together by *your code* into a fixed, auditable sequence.
> - **Structured outputs** (`output_config` + JSON Schema) turn model output into data your code can branch on reliably.
> - The **evaluator–optimizer** gate (draft → review → revise) is a cheap, powerful reliability boost.
> - You trade the agent's flexibility for **predictability** — the right call when the steps are known in advance.

# Form Factor 4 — The Agent

> 💡 **Key Term — Agent:** An LLM running in a loop with tools. Unlike a workflow, *the model* decides which tool to call, inspects the result, and decides what to do next — repeating until the task is done. You provide the tools and the goal; the model provides the trajectory.

We build this with the **`claude-agent-sdk`** — the same runtime that powers Claude Code. We give the agent two tools (search the docs, open a support ticket) and let it work out the rest.

> 💡 **Key Insight — Workflow vs. Agent:** In Form Factor 3 *we* wrote `classify → retrieve → draft → review`. Here we write none of that. We hand the model a goal and some tools, and it chooses the sequence itself. More flexible, less predictable — use it when the path can't be fully planned in advance.

![The four faculties of an agent](../images/ff3_agent.png)


> **Note:** The `claude-agent-sdk` bundles a Node.js-based runtime and reads `ANTHROPIC_API_KEY` from your environment. Make sure Node.js is installed and the package is available (see the setup cell). The cell below imports the SDK and lets its async code run inside Jupyter.

In [44]:
import nest_asyncio
nest_asyncio.apply()  # allow the SDK's async runtime to work inside a notebook

from claude_agent_sdk import (
    query,
    tool,
    create_sdk_mcp_server,
    ClaudeAgentOptions,
    AssistantMessage,
    ResultMessage,
)
print("claude-agent-sdk imported ✓")

claude-agent-sdk imported ✓


## 4.1 Give the Agent Tools

A tool is just a Python function the model can choose to call. We define each with the `@tool` decorator. The description tells the model *when* to use it — that guidance is how the agent reasons about tool choice.

**Tool 1 — search the docs** (reuses the RAG retriever from Form Factor 2).

In [45]:
@tool(
    "search_docs",
    "Search the Acme Cloud documentation. Use this whenever the user asks about Acme "
    "Cloud's plans, pricing, limits, or features.",
    {"query": str},
)
async def search_docs(args):
    hits = retrieve(args["query"], k=3)            # reuse the RAG retriever from Form Factor 2
    text = "\n".join(f"[{i + 1}] {doc}" for i, (doc, _) in enumerate(hits))
    return {"content": [{"type": "text", "text": text}]}

**Tool 2 — open a support ticket.** A tool can *act*, not just read. Its description tells the agent to use it only for genuine escalations.

In [46]:
@tool(
    "create_support_ticket",
    "Open a support ticket. Use this only when the user asks to escalate, or reports an "
    "urgent, unresolved problem.",
    {"summary": str, "priority": str},
)
async def create_support_ticket(args):
    ticket_id = f"ACME-{abs(hash(args['summary'])) % 10000:04d}"
    msg = f"Created ticket {ticket_id} (priority={args['priority']}): {args['summary']}"
    return {"content": [{"type": "text", "text": msg}]}

**Register the tools** on an in-process MCP server, which the agent will connect to.

> 💡 **What is an MCP server, and why wrap tools in one?** **MCP — the Model Context Protocol** — is an open standard for exposing tools, data, and prompts to LLMs through a uniform interface. Rather than every framework inventing its own way to describe and invoke a tool, the model's runtime talks to an **MCP server** that advertises a list of tools and executes them on request.
>
> The `claude-agent-sdk` speaks MCP natively, so the way you hand it custom Python functions is to put them on an MCP server. `create_sdk_mcp_server(...)` builds an **in-process** one — it runs inside this very Python process (no separate service, no network hop), and the agent calls our `search_docs` / `create_support_ticket` functions directly.

> 💡 **The alternative — and why MCP earns its keep.** The other option is to **hand the model a plain list of tool definitions** (conceptually like the raw Messages API's own `tools=[...]` parameter): just functions, no server wrapper. For a one-off script that's marginally less ceremony. But registering the *same* functions as an MCP server buys four things a bare list doesn't:
>
> - **Reusability across clients.** An MCP server is a self-contained capability. The exact same `search_docs` / `create_support_ticket` server can be plugged into Claude Code, Claude Desktop, a different agent, or a teammate's project **without touching the functions**. A local tool list is welded to this one app.
> - **Portability, in-process → networked.** Today it runs in-process for zero latency. The very same interface can later be hosted as a separate process or a remote service (even written in another language) with almost no change to the agent code — the protocol is the contract.
> - **Composability.** Because it's a shared standard, you can mix your server with off-the-shelf MCP servers (GitHub, Slack, a database, the filesystem) under one uniform interface, instead of hand-adapting each integration.
> - **Discovery, scoping & governance.** The server *advertises* its tools and their schemas, so the runtime can list them, log their use, and gate them. That's precisely what `allowed_tools=["mcp__acme__search_docs", …]` does — it names tools in MCP's `mcp__<server>__<tool>` form to pre-approve and tightly scope what the agent may call.
>
> **Rule of thumb:** for a quick, self-contained, single-app tool, a plain tool list is perfectly fine. The moment a tool might be **reused, shared, served remotely, or governed**, an MCP server is the cleaner home — and because the SDK speaks MCP natively, you get all of the above here for almost no extra code.

In [47]:
acme_tools = create_sdk_mcp_server(name="acme", version="1.0.0",
                                   tools=[search_docs, create_support_ticket])
print("Registered tools: search_docs, create_support_ticket")

Registered tools: search_docs, create_support_ticket


## 4.2 Run the Agent

The helper below streams the agent's messages and prints both its **tool calls** (the decisions it makes) and its **text** (what it says). Watching the tool calls is the best way to *see* the agent reasoning.

In [48]:
async def run_agent(prompt: str, options: ClaudeAgentOptions):
    """Stream an agent run, printing tool calls and assistant text as they arrive."""
    async for message in query(prompt=prompt, options=options):
        if isinstance(message, AssistantMessage):
            for block in message.content:
                if hasattr(block, "text"):                 # a TextBlock
                    print(block.text)
                elif hasattr(block, "name"):               # a ToolUseBlock
                    print(f"  🔧 {block.name}({dict(block.input)})")
        elif isinstance(message, ResultMessage):
            print("\n✓ agent finished")

We configure the agent with our two tools and a system prompt, and keep it hermetic with `setting_sources=[]` (so it ignores any local project/user settings).

**`ClaudeAgentOptions` — the agent's configuration.** Everything that shapes the run is set here and handed to `query()`:

- **`model`** — which Claude model runs the agent loop (`claude-opus-4-8`).
- **`system_prompt`** — the agent's standing instructions: its role, how it should use its tools, and the rules to follow.
- **`mcp_servers`** — a map of name → **in-process MCP server**. Our tools (`search_docs`, `create_support_ticket`) live on the `acme_tools` server we built with `create_sdk_mcp_server`; registering it here is what makes those Python functions *callable by the model*. (MCP — the Model Context Protocol — is the standard interface the SDK uses to expose tools to a model.)
- **`allowed_tools`** — an explicit allowlist of the exact tools the agent may call, written in MCP's `mcp__<server>__<tool>` form (`mcp__acme__search_docs`, `mcp__acme__create_support_ticket`). It pre-approves these tools so they run **without a permission prompt**, and just as importantly it **scopes** the agent — it will not reach for anything not on the list.
- **`setting_sources=[]`** — keeps the run hermetic by ignoring any local project/user configuration.

**Why both `mcp_servers` and `allowed_tools`?** They answer two different questions. `mcp_servers` makes a tool *exist* for the agent; `allowed_tools` decides which of the existing tools it is *permitted to use*. You need both — a registered tool that isn't allow-listed can't be called, and an allow-listed name that no registered server provides resolves to nothing.

In [49]:
support_agent_options = ClaudeAgentOptions(
    model=MODEL,
    system_prompt=(
        "You are the Acme Cloud support agent. You have exactly two tools: search_docs and "
        "create_support_ticket. Use search_docs to ground every factual claim. Only open a "
        "ticket if the user clearly needs escalation. Cite docs with [n]."
    ),
    mcp_servers={"acme": acme_tools},
    allowed_tools=["mcp__acme__search_docs", "mcp__acme__create_support_ticket"],
    setting_sources=[],
)

Now give it a request that needs *both* tools — answer a question (search) **and** escalate (ticket). Notice we never tell it the order; it decides.

In [50]:
await run_agent(
    "I'm on the Pro plan and keep hitting rate limits right before our launch tomorrow. "
    "What are my options, and can you escalate this for me?",
    support_agent_options,
)

warn: CPU lacks AVX support, strange crashes may occur. Reinstall Bun or use *-baseline build:
  https://github.com/oven-sh/bun/releases/download/bun-v1.3.14/bun-darwin-x64-baseline.zip


I'll load my tools and look up the relevant documentation.
  🔧 ToolSearch({'query': 'select:mcp__acme__search_docs,mcp__acme__create_support_ticket', 'max_results': 5})
I'll search the docs for rate limit options and plan details before answering.
  🔧 mcp__acme__search_docs({'query': 'Pro plan rate limits'})
  🔧 mcp__acme__search_docs({'query': 'how to increase rate limits or request a limit raise'})
Here are your options, grounded in the Acme Cloud docs:

**Your current limit**
- The Pro plan allows **1,000 requests per minute** [1]. If you're hitting this consistently, you have a few paths.

**Options**
1. **Reduce request volume** — batch or cache calls, add client-side retry/backoff so bursts don't slam the limit. This is the only option that helps *immediately* without a plan change.
2. **Upgrade to Enterprise** — Enterprise rate limits are **negotiated per contract** rather than fixed, so you can secure a higher ceiling for the launch [1]. Upgrades take effect immediately and are

> 💡 **Key Insight — The agent chose the path:** No `if`/`for` in our code decided to search the docs *and then* open a ticket — the model did, by reading each tool result and deciding the next move. That is the leap from workflow to agent.
>
> For a back-and-forth conversation with the same agent (memory across turns), use `ClaudeSDKClient` as an async context manager instead of `query()`.

> ### 📌 Key Takeaways — Form Factor 4
> - An **agent** is an LLM in a tool-calling loop; *the model* chooses which tools to call and when.
> - The **`claude-agent-sdk`** runs that loop for you; you supply tools (`@tool` + `create_sdk_mcp_server`), a goal, and `allowed_tools`.
> - **Tool descriptions are prompts** — they're how the model decides when each tool applies.
> - Use an agent when the path **can't be planned up front**; accept less predictability for more flexibility.

# Form Factor 5 — The Autonomous Agent

> 💡 **Key Term — Autonomous agent:** An agent equipped with tools that change the world — here, the ability to **write files and run shell commands**. It can build something durable (a script, a config, a small program), test it, and fix it, all on its own.

This is the top rung: the agent doesn't just *answer*, it *produces a working automation*. We'll ask it to build a small, **re-runnable command-line tool** that triages any batch of support tickets — then run it, verify it, and run it again on a *fresh* batch to prove it keeps working. It does all of this with the agent SDK's built-in `Write`, `Read`, `Edit`, and `Bash` tools.

> 💡 **What makes it an *automation*, not just a script?** The agent doesn't produce a one-shot answer — it produces a **reusable tool**: a parametrized CLI that takes an input file, so it can be pointed at *new* data, scheduled (cron / Task Scheduler), or dropped into a pipeline. That's the difference between "ran some code once" and "built something that keeps doing the job."

> ⚠️ **Safety — this agent runs code:** With file-write and shell access plus `permission_mode="bypassPermissions"`, the agent acts without asking. That is the point of autonomy, but it means you must **scope it tightly**. We confine it to a throwaway `cwd` sandbox folder and give it a single, well-defined task. Never point an autonomous agent at a directory you care about, or at a production system, without a real permission policy.

**Set up a sandbox** and drop in a small, *real* sample of customer-support tickets — pulled from the [`Tobi-Bueck/customer-support-tickets`](https://huggingface.co/datasets/Tobi-Bueck/customer-support-tickets) dataset on Hugging Face — for the agent to build an automation around. We fetch only a tiny slice (no full download) and fall back to an inline sample if the network is unavailable, so this cell always runs.

We write **two** ticket files: `tickets_batch1.csv` (the initial data the agent builds against) and `tickets_batch2.csv` (a second "new arrivals" batch). That second file is how we'll prove the agent built a *real automation* — a tool that re-runs on input it has never seen — rather than a one-off script hard-wired to one file.

In [51]:
import os, csv, json, pathlib

SANDBOX = pathlib.Path("./ff5_sandbox").resolve()
SANDBOX.mkdir(exist_ok=True)

# Stock the sandbox with real tickets (reusing `load_support_tickets` from §3.1),
# mapped onto the columns our automation will work over:
#   queue -> category    priority -> priority    subject -> message
# We write TWO files: an initial batch the agent builds against, and a second
# "new arrivals" batch we use later to prove the tool re-runs on fresh input.
def write_batch(path, tickets, start_id=1):
    rows = [
        {"id": start_id + i, "category": t["queue"], "priority": t["priority"], "message": t["subject"]}
        for i, t in enumerate(tickets)
    ]
    with open(path, "w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=["id", "category", "priority", "message"])
        writer.writeheader()
        writer.writerows(rows)
    return len(rows)


tickets = load_support_tickets(12)
half = max(1, len(tickets) // 2)
n1 = write_batch(SANDBOX / "tickets_batch1.csv", tickets[:half], start_id=1)
n2 = write_batch(SANDBOX / "tickets_batch2.csv", tickets[half:], start_id=half + 1)

print("Sandbox ready at", SANDBOX)
print(f"Wrote {n1} tickets -> tickets_batch1.csv   and   {n2} tickets -> tickets_batch2.csv")
print("Files:", [p.name for p in SANDBOX.iterdir()])

Sandbox ready at /Users/richmondalake/Desktop/agent_memory_course/part5/ai_maturity_form_factors/notebook/ff5_sandbox
Wrote 6 tickets -> tickets_batch1.csv   and   6 tickets -> tickets_batch2.csv
Files: ['tickets_batch2.csv', 'tickets_batch1.csv', 'report_batch1.json', 'triage.py', 'report_batch2.json']


**Configure the builder agent** with the file/shell tools, pointed at the sandbox.

In [52]:
builder_options = ClaudeAgentOptions(
    model=MODEL,
    cwd=str(SANDBOX),                                   # all file/shell ops stay in the sandbox
    allowed_tools=["Read", "Write", "Edit", "Bash"],    # the tools that let it build & run code
    permission_mode="bypassPermissions",                # autonomous: no human approval prompts
    max_turns=30,
    setting_sources=[],
    system_prompt="You are an automation engineer. Build the smallest correct, reusable solution, then verify it runs on every input you are given.",
)

**Turn it loose.** The agent will build the CLI, run it on the first batch, then re-run it on the second batch — fixing its own errors along the way, autonomously.

In [53]:
await run_agent(
    "You are building a reusable automation in the current directory. It contains two CSV files, "
    "`tickets_batch1.csv` and `tickets_batch2.csv`, each with columns id,category,priority,message.\n"
    "1. Write a minimal, well-documented Python CLI `triage.py` (standard library only) that:\n"
    "   - accepts `--input <path>` (the tickets CSV) and `--output <path>` (default `report.json`);\n"
    "   - computes two breakdowns -- messages per category and per priority -- each sorted by count descending;\n"
    "   - writes both breakdowns to the output path as JSON;\n"
    "   - validates its input and exits non-zero with a clear message if the file is missing or its columns are wrong;\n"
    "   - uses NO hardcoded paths, so it can be scheduled or reused on any batch.\n"
    "2. Run it on the first batch:  python triage.py --input tickets_batch1.csv --output report_batch1.json\n"
    "3. Then prove it re-runs on NEW data:  python triage.py --input tickets_batch2.csv --output report_batch2.json\n"
    "4. If anything errors, fix it and re-run until both batches succeed.\n"
    "5. Confirm what you built, show both report files, and give the cron line you'd use to schedule it.",
    builder_options,
)

warn: CPU lacks AVX support, strange crashes may occur. Reinstall Bun or use *-baseline build:
  https://github.com/oven-sh/bun/releases/download/bun-v1.3.14/bun-darwin-x64-baseline.zip


I'll build this in the current directory. Let me first check the CSV files to understand the data.
  🔧 Bash({'command': 'pwd && ls -la && echo "---batch1---" && head -5 tickets_batch1.csv && echo "---batch2---" && head -5 tickets_batch2.csv', 'description': 'List directory and preview CSV files'})
There's an existing `triage.py` and report files, but I'll write a fresh, correct version to be sure it meets all the requirements.
  🔧 Write({'file_path': '/Users/richmondalake/Desktop/agent_memory_course/part5/ai_maturity_form_factors/notebook/ff5_sandbox/triage.py', 'content': '#!/usr/bin/env python3\n"""triage.py -- summarize a support-ticket CSV into category/priority breakdowns.\n\nReads a tickets CSV with columns: id,category,priority,message\nWrites a JSON report with two count breakdowns (each sorted by count desc):\n  - "by_category"\n  - "by_priority"\n\nStandard library only. No hardcoded paths -- safe to schedule/reuse on any batch.\n\nUsage:\n    python triage.py --input tickets

## 5.1 Inspect the Artifacts

The agent didn't just describe a solution — it left a **reusable tool** and its outputs on disk. Let's read back the CLI and the report it produced for *each* batch, confirming the same automation handled two different inputs.

In [54]:
print("Sandbox now contains:", [p.name for p in SANDBOX.iterdir()], "\n")

tool_path = SANDBOX / "triage.py"
if tool_path.exists():
    print("=== triage.py (the reusable automation) ===")
    print(tool_path.read_text())

for report in ["report_batch1.json", "report_batch2.json"]:
    p = SANDBOX / report
    if p.exists():
        print(f"\n=== {report} ===")
        print(p.read_text())

Sandbox now contains: ['tickets_batch2.csv', 'tickets_batch1.csv', 'report_batch1.json', 'triage.py', 'report_batch2.json'] 

=== triage.py (the reusable automation) ===
#!/usr/bin/env python3
"""Triage support tickets into category and priority breakdowns.

Reads a tickets CSV (columns: id,category,priority,message) and writes a JSON
report containing two breakdowns -- counts per category and per priority --
each sorted by count descending.

Standard library only. No hardcoded paths, so it can be scheduled (cron) or
reused on any batch via --input/--output.

Usage:
    python triage.py --input tickets_batch1.csv --output report_batch1.json
"""

import argparse
import csv
import json
import sys
from collections import Counter

# Columns the input CSV must contain (order-independent).
REQUIRED_COLUMNS = {"id", "category", "priority", "message"}


def parse_args(argv=None):
    """Define and parse command-line arguments."""
    parser = argparse.ArgumentParser(
        description="Summa

> 💡 **Key Insight — From answering to building:** Every earlier form factor produced *text*. This one produced a *reusable program* that keeps working long after the agent stops — we ran it on a second batch it had never seen, and it just worked. That is the defining capability of an autonomous agent, and the reason it demands the most care.

> ### 📌 Key Takeaways — Form Factor 5
> - An **autonomous agent** uses world-changing tools (`Write`, `Bash`, `Edit`) to produce **durable, reusable artifacts** — here, a parametrized CLI you can schedule — not just answers.
> - It runs a **build → run → fix** loop on its own, verifying its own work; we confirmed the result is a genuine *automation* by re-running it on fresh input.
> - This power demands guardrails: a **sandbox `cwd`**, a tight task, and a real permission policy in production (`bypassPermissions` is for demos).
> - Top of the ladder — reach for it only when the job is to *build something reusable*, not just respond.

# Where to Next?

You climbed all five rungs of the ladder, each adding one capability:

| # | Form Factor | New capability | Reach for it when… |
|---|-------------|----------------|--------------------|
| 1 | **Chatbot** | Generate text | The answer is in the model's general knowledge |
| 2 | **RAG** | Ground in your data | Answers must reflect *your* documents or data |
| 3 | **Workflow** | Reliable multi-step pipelines | The steps are known and you want predictability |
| 4 | **Agent** | Model-chosen tool use | The path can't be planned up front |
| 5 | **Autonomous agent** | Write & run code | The job is to *build* something, not just answer |

> 💡 **Key Insight — Climb only as high as you need:** Lower rungs are cheaper, faster, and more predictable. Start at the simplest form factor that solves your problem, and move up only when a real limitation forces you to.

**Keep learning:**

- **[Building Effective Agents](https://www.anthropic.com/research/building-effective-agents)** — Anthropic's guide to workflows vs. agents
- **[Claude Agent SDK docs](https://code.claude.com/docs/en/agent-sdk/overview)** — build custom agents (Form Factors 4–5)
- **[Claude API docs](https://platform.claude.com/docs/en/api/overview)** — the Messages API behind Form Factors 1–3
- **[Structured outputs](https://platform.claude.com/docs/en/build-with-claude/structured-outputs)** — the JSON-schema technique from the workflow
- **[Oracle AI Vector Search](https://docs.oracle.com/en/database/oracle/oracle-database/26/vecse/)** — vectors and hybrid search in Oracle